# MERFISH codebook construction — Python translation

This notebook translates the **codebook construction** subset of the ZhuangLab/MERFISH_analysis MATLAB repository into Python.

**Repository:** https://github.com/ZhuangLab/MERFISH_analysis
**Original publications:** Chen *et al.* 2015 (Science), Moffitt *et al.* 2016 (PNAS).

## Scope

The full repository contains 86 MATLAB files. They fall into four parts:

| GitHub folder | Purpose | Translated here? |
|---|---|---|
| `codes/` | Binary barcode (codeword) construction | **Yes — 5 files** |
| `fileIO/` | Reading and writing codebook CSVs and proprietary binary microscopy formats | **Yes — 2 files** (`LoadCodebook`, `WriteCodebook` only) |
| `example_scripts/` | Top-level demo scripts | **Yes — 1 file** (`code_construction_script`) |
| `analysis/` | Image decoding from raw microscopy (SLURM cluster jobs) | No — needs raw fluorescence images |
| `probe_construction/` | Designs DNA oligonucleotide probes for new MERFISH experiments | No — designs experiments, not analysis |
| `startup/` | MATLAB path setup | No — irrelevant outside MATLAB |
| `deprecated/` | Old versions | No |

**Why only these 8 files:** every other file in the repo either (a) reads proprietary binary microscopy files we do not have, (b) decodes raw images we do not have, (c) designs DNA probes for new wet-lab experiments, or (d) uses MATLAB-only Communications Toolbox functions (`hammgen`, `gen2par`, `encode`) that have no direct Python equivalent.

The 8 translated files operate on the **same conceptual level as the thesis dataset**: the binary barcodes that encode the 155 genes Moffitt et al. used to label cells.

## Translated files

| # | MATLAB file | Python function | What it does |
|---|---|---|---|
| 1 | `codes/GenerateExtendedHammingWords.m` | `generate_extended_hamming_words` | Builds Extended Hamming codewords (parity matrix + generator), used to make MHD4 barcodes. |
| 2 | `codes/GenerateSurroundingCodewords.m` | `generate_surrounding_codewords` | Lists all codewords at exactly Hamming distance *d* from a given codeword. |
| 3 | `codes/SECDEDCorrectableWords.m` | `secded_correctable_words` | Lists the codewords that SECDED would correct to the given codeword (HD = 1). |
| 4 | `codes/GenSECDED.m` | `gen_secded` | Generates SECDED codewords for a given length and on-bit count. |
| 5 | `codes/CodebookToMap.m` | `codebook_to_map` | Builds a dict mapping each codeword (and optionally its correctable neighbours) to a gene name. |
| 6 | `fileIO/LoadCodebook.m` | `load_codebook` | Reads a MERFISH codebook CSV (header + name/id/barcode rows). |
| 7 | `fileIO/WriteCodebook.m` | `write_codebook` | Writes a MERFISH codebook CSV. |
| 8 | `example_scripts/code_construction_script.m` | (the demo cell at the bottom) | Runs MHD4 + MHD2 construction end-to-end; verifies code properties. |


## 0. Imports


In [ ]:
import numpy as np
import pandas as pd
from itertools import combinations
from typing import Iterable, List, Dict, Tuple, Optional, Callable


## 1. `generate_extended_hamming_words` — `codes/GenerateExtendedHammingWords.m`

**What it does:** generates every Extended Hamming codeword for a given number of data bits.

**Why it matters in MERFISH:** the 16-bit MHD4 codebook used in the 2015 *Science* paper and many subsequent MERFISH studies is built by taking all Extended Hamming codewords and keeping only those with Hamming weight 4. This guarantees a minimum Hamming distance of 4 between any two valid codewords, allowing single-bit error correction and double-bit error detection.

The MATLAB code uses Communications Toolbox helpers (`hammgen`, `gen2par`). I re-implement the math directly using numpy so the Python function is self-contained.


In [ ]:
def generate_extended_hamming_words(num_data_bits: int) -> Tuple[np.ndarray, np.ndarray, int]:
    """Translate of codes/GenerateExtendedHammingWords.m.

    Returns all Extended Hamming codewords for the given number of data bits.

    Parameters
    ----------
    num_data_bits : int
        Number of information (message) bits k.

    Returns
    -------
    words : (2**k, n) ndarray of {0,1}
        Every codeword in the (n, k) Extended Hamming code.
    generator : (k, n) ndarray of {0,1}
        Generator matrix used to map information words to codewords.
    num_parity_bits : int
        Number of parity bits added (n - k).

    Notes
    -----
    The original MATLAB code uses `hammgen` and `gen2par` from the Communications
    Toolbox. We re-implement directly:

      1. Choose m so that 2**m - m - 1 >= k  (m parity bits before extension).
      2. Build H_ham as the m x (2**m - 1) Hamming parity-check matrix whose
         columns are the binary expansions of 1, 2, ..., 2**m - 1.
      3. Extend by appending a zero column and an all-ones row, yielding the
         (m+1) x 2**m parity-check matrix H of the Extended Hamming code.
      4. Find a basis of the null space of H over GF(2) — this is the generator
         matrix G of shape (k_full, n) where k_full = n - rank(H).
      5. Enumerate all 2**k_full information words and encode them, then keep
         the first 2**k via shortening if needed.

    For k=11 this produces the (16, 11) Extended Hamming code that underlies
    the MHD4 codebook used in MERFISH (140 weight-4 codewords, min HD = 4).
    """
    if num_data_bits < 1:
        raise ValueError("num_data_bits must be >= 1")

    k = num_data_bits

    # 1. Smallest m with 2**m - m - 1 >= k
    m = 1
    while (2**m - m - 1) < k:
        m += 1

    n_ham = 2**m - 1
    n = n_ham + 1                      # extended length

    # 2. Hamming parity-check matrix: columns are 1..n_ham in binary
    H_ham = np.array(
        [[(j >> (m - 1 - i)) & 1 for j in range(1, n_ham + 1)] for i in range(m)],
        dtype=np.uint8,
    )

    # 3. Extend to (m+1) x n
    H = np.zeros((m + 1, n), dtype=np.uint8)
    H[:m, :n_ham] = H_ham
    H[m, :] = 1                        # overall parity row

    # 4. Find null space of H over GF(2) via RREF, returning a generator G
    G_full = _gf2_nullspace(H)         # shape (k_full, n)
    k_full = G_full.shape[0]

    if k > k_full:
        raise ValueError(
            f"Cannot encode {k} data bits in this Extended Hamming code (max {k_full})."
        )

    # Shorten by taking the first k generator rows
    G = G_full[:k, :]

    # 5. Enumerate all 2**k information words and encode
    info = np.array(
        [[(d >> (k - 1 - i)) & 1 for i in range(k)] for d in range(2**k)],
        dtype=np.uint8,
    )
    words = (info @ G) % 2

    return words.astype(np.uint8), G.astype(np.uint8), int(n - k)


def _gf2_rref(M: np.ndarray) -> np.ndarray:
    """Reduced row-echelon form of a binary matrix over GF(2)."""
    A = M.copy().astype(np.uint8)
    rows, cols = A.shape
    r = 0
    for c in range(cols):
        if r >= rows:
            break
        pivot = None
        for i in range(r, rows):
            if A[i, c] == 1:
                pivot = i
                break
        if pivot is None:
            continue
        if pivot != r:
            A[[r, pivot]] = A[[pivot, r]]
        for i in range(rows):
            if i != r and A[i, c] == 1:
                A[i] = (A[i] + A[r]) % 2
        r += 1
    return A


def _gf2_nullspace(H: np.ndarray) -> np.ndarray:
    """Basis of the null space of H over GF(2), returned as a (k, n) matrix
    whose rows span the kernel: every row x satisfies H @ x = 0 (mod 2).

    Implementation: reduce H to RREF, identify pivot vs free columns, then
    construct one basis vector per free column."""
    R = _gf2_rref(H)
    rows, n = R.shape

    # Identify pivot columns
    pivot_cols = []
    pivot_row_for_col = {}
    for i in range(rows):
        for j in range(n):
            if R[i, j] == 1:
                pivot_cols.append(j)
                pivot_row_for_col[j] = i
                break
    pivot_set = set(pivot_cols)
    free_cols = [j for j in range(n) if j not in pivot_set]

    basis = []
    for free in free_cols:
        x = np.zeros(n, dtype=np.uint8)
        x[free] = 1
        # For each pivot column p, the basis vector x must satisfy
        # x[p] = R[pivot_row_for_col[p], free] (so that R @ x = 0)
        for p, prow in pivot_row_for_col.items():
            x[p] = R[prow, free]
        basis.append(x)

    if not basis:
        return np.zeros((0, n), dtype=np.uint8)
    return np.array(basis, dtype=np.uint8)


## 2. `generate_surrounding_codewords` — `codes/GenerateSurroundingCodewords.m`

**What it does:** given a codeword and a Hamming distance *d*, lists every codeword exactly distance *d* away from it (i.e. every codeword that differs in exactly *d* bit positions).

**Why it matters:** used to enumerate all single-bit and double-bit errors of every valid barcode. This is how the decoder maps an observed (possibly corrupted) measurement back to the nearest valid codeword.


In [ ]:
def generate_surrounding_codewords(codeword: Iterable[int],
                                   hamm_dist: int) -> List[np.ndarray]:
    """Translate of codes/GenerateSurroundingCodewords.m.

    Returns all binary codewords at exactly Hamming distance `hamm_dist`
    from `codeword`.

    Parameters
    ----------
    codeword : 1-D iterable of {0,1}
        The reference codeword.
    hamm_dist : int
        The exact Hamming distance for the returned codewords.

    Returns
    -------
    list of 1-D ndarray
        Each entry is a codeword with exactly `hamm_dist` bits flipped
        relative to the input.
    """
    cw = np.asarray(list(codeword), dtype=np.uint8)
    if cw.ndim != 1:
        raise ValueError("codeword must be 1-D")
    n = cw.shape[0]
    out = []
    for flip_idx in combinations(range(n), hamm_dist):
        new_cw = cw.copy()
        new_cw[list(flip_idx)] ^= 1     # toggle those bits
        out.append(new_cw)
    return out


## 3. `secded_correctable_words` — `codes/SECDEDCorrectableWords.m`

**What it does:** lists the codewords that SECDED (Single Error Correction, Double Error Detection) would correct to the input codeword. This is exactly the set of codewords at Hamming distance 1.

**Why it matters:** used by the decoder to build a lookup table of correctable single-bit errors for each valid barcode.

The MATLAB function is a one-liner — it just calls `GenerateSurroundingCodewords(codeword, 1)`. The Python version does the same.


In [ ]:
def secded_correctable_words(codeword: Iterable[int]) -> List[np.ndarray]:
    """Translate of codes/SECDEDCorrectableWords.m.

    Returns all codewords that SECDED would correct to the given codeword
    (i.e. all codewords at Hamming distance 1).
    """
    return generate_surrounding_codewords(codeword, hamm_dist=1)


## 4. `gen_secded` — `codes/GenSECDED.m`

**What it does:** generates SECDED codewords for a given total length, number of data bits, and required number of on-bits.

**Why it matters:** SECDED is an alternative encoding scheme to MHD4. The original code has a special case for 12-bit / 4-on-bit codewords. The Python version handles the general case the same way.

The MATLAB function uses the Communications Toolbox `encode(...,'hamming/binary')`. I implement Hamming encoding by hand using the same parity-check construction as in function 1.


In [ ]:
def gen_secded(num_letters: int,
               num_data_bits: int,
               on_bits: Optional[int] = None) -> np.ndarray:
    """Translate of codes/GenSECDED.m.

    Generate SECDED codewords (a Hamming code extended with one overall
    parity bit, so the minimum distance becomes 4).

    Parameters
    ----------
    num_letters : int
        Total length n of each codeword.
    num_data_bits : int
        Number of information bits k.
    on_bits : int or None
        If given, restrict output to codewords with exactly this many 1s.
        If None, return all SECDED codewords with weight neither very small
        (<=4) nor very large (>= n-4), matching the MATLAB filter.

    Returns
    -------
    (M, num_letters) ndarray of {0,1}
        The selected SECDED codewords.
    """
    if num_letters < num_data_bits:
        raise ValueError("data bits cannot exceed total bits")
    if on_bits is not None and num_data_bits < on_bits:
        raise ValueError("on_bits cannot exceed data bits")

    # SECDED = Extended Hamming. Reuse the same routine.
    n = num_letters
    m = 1
    while (2**m) < n:
        m += 1
    # Build (m+1) x n parity-check matrix using leftmost n-1 columns of the
    # Hamming(2**m-1, 2**m-1-m) code, plus the overall parity row.
    full_ham = np.array(
        [[(j >> (m - 1 - i)) & 1 for j in range(1, 2**m)] for i in range(m)],
        dtype=np.uint8,
    )
    H_ham = full_ham[:, :n - 1]
    H = np.zeros((m + 1, n), dtype=np.uint8)
    H[:m, :n - 1] = H_ham
    H[m, :] = 1                             # overall parity row

    G_full = _gf2_nullspace(H)              # (k_full, n)
    k_full = G_full.shape[0]
    if num_data_bits > k_full:
        raise ValueError(
            f"{num_data_bits} data bits requested but this length supports only {k_full}"
        )
    G = G_full[:num_data_bits, :]

    info = np.array(
        [[(d >> (num_data_bits - 1 - i)) & 1 for i in range(num_data_bits)]
         for d in range(2**num_data_bits)],
        dtype=np.uint8,
    )
    secded = (info @ G) % 2

    weights = secded.sum(axis=1)
    if on_bits is not None:
        keep = weights == on_bits
    else:
        keep = (weights > 4) & (weights < num_letters - 4)
    return secded[keep].astype(np.uint8)


## 5. `codebook_to_map` — `codes/CodebookToMap.m`

**What it does:** turns a list of (gene, barcode) pairs into a dictionary that maps each codeword (and optionally every error-correctable neighbour of each codeword) to the corresponding gene name.

**Why it matters:** this dictionary is the decoder's lookup table. When the imaging pipeline observes a (possibly noisy) binary readout, it looks up the readout in the map to identify which gene was detected.


In [ ]:
def codebook_to_map(codebook: List[Dict],
                    err_corr_func: Optional[Callable] = None,
                    key_type: str = "int",
                    map_contents: str = "all") -> Dict:
    """Translate of codes/CodebookToMap.m.

    Build a dict mapping codewords to gene names.

    Parameters
    ----------
    codebook : list of dict
        Each entry must have keys 'name' (gene name) and 'barcode'
        (binary string or 1-D array of {0,1}).
    err_corr_func : callable or None
        If provided, called as err_corr_func(barcode_array) and must return
        a list of additional codewords (the "correctable" neighbours) that
        should map to the same gene. E.g. `secded_correctable_words`.
    key_type : {'int','binStr'}
        How to encode the codeword as a dict key.
        'int'    -> integer (left-MSB binary -> int)
        'binStr' -> string of '0'/'1'
    map_contents : {'exact','correctable','all'}
        Which entries to keep when err_corr_func is given:
        'exact'       -> only the original codewords
        'correctable' -> only the corrected neighbours
        'all'         -> both
        When err_corr_func is None this argument is ignored.

    Returns
    -------
    dict
    """
    def _bc_to_array(bc):
        if isinstance(bc, str):
            return np.array([int(c) for c in bc.strip() if c in "01"], dtype=np.uint8)
        return np.asarray(bc, dtype=np.uint8).flatten()

    def _key(arr):
        if key_type == "int":
            v = 0
            for b in arr:
                v = (v << 1) | int(b)
            return v
        elif key_type == "binStr":
            return "".join(str(int(b)) for b in arr)
        else:
            raise ValueError("key_type must be 'int' or 'binStr'")

    exact_map: Dict = {}
    for entry in codebook:
        bc = _bc_to_array(entry["barcode"])
        exact_map[_key(bc)] = entry["name"]

    if err_corr_func is None:
        return exact_map

    correctable_map: Dict = {}
    for entry in codebook:
        bc = _bc_to_array(entry["barcode"])
        for neighbour in err_corr_func(bc):
            correctable_map[_key(np.asarray(neighbour, dtype=np.uint8))] = entry["name"]

    if map_contents == "exact":
        return exact_map
    if map_contents == "correctable":
        return correctable_map
    if map_contents == "all":
        return {**exact_map, **correctable_map}
    raise ValueError("map_contents must be 'exact', 'correctable', or 'all'")


## 6. `load_codebook` — `fileIO/LoadCodebook.m`

**What it does:** reads a MERFISH codebook CSV. The file format is:

```
version, 1.0
codebook_name, M3E1
bit_names, RS0015, RS0083, RS0095, ...
name, id, barcode
Gad1, ENSMUST00000xxx, 1 0 1 0 0 1 0 1 0 0 0 0 0 0 0 0
Slc17a6, ENSMUST00000yyy, 0 1 0 1 0 0 1 0 0 0 0 0 1 0 0 0
...
```

The header is name/value pairs until a row of literally `name, id, barcode`. Every row after that is a (gene name, transcript id, barcode) triple. The barcode is a space-separated string of 0/1.


In [ ]:
def load_codebook(codebook_path: str,
                  verbose: bool = True) -> Tuple[List[Dict], Dict]:
    """Translate of fileIO/LoadCodebook.m.

    Parameters
    ----------
    codebook_path : str
        Path to the codebook CSV.
    verbose : bool

    Returns
    -------
    codebook : list of dict
        Each dict has 'name', 'id', 'barcode' (the barcode as a string of '0'/'1').
    header : dict
        Header fields as parsed (e.g. version, codebook_name, bit_names).
    """
    if verbose:
        print(f"-- Loading codebook from: {codebook_path}")

    header: Dict = {}
    rows = []

    with open(codebook_path, "r") as f:
        # Read header
        while True:
            line = f.readline()
            if not line:
                raise ValueError("codebook ended before barcode section")
            parts = [p.strip() for p in line.strip().split(",")]
            if set(parts) >= {"name", "id", "barcode"} and len(parts) == 3:
                # this is the barcode-section divider
                break
            if len(parts) == 2:
                header[parts[0]] = parts[1]
            elif len(parts) > 2:
                header[parts[0]] = parts[1:]

        # Validate header
        if "version" not in header or "bit_names" not in header:
            raise ValueError("codebook missing required header fields 'version'/'bit_names'")

        # Read body
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = [p.strip() for p in line.split(",")]
            if len(parts) < 3:
                continue
            name, id_, bc_raw = parts[0], parts[1], parts[2]
            bc = "".join(c for c in bc_raw if c in "01")
            rows.append({"name": name, "id": id_, "barcode": bc})

    if verbose:
        for k, v in header.items():
            shown = ", ".join(v) if isinstance(v, list) else v
            print(f"   {k}: {shown}")
        print(f"   ...loaded {len(rows)} barcodes")

    return rows, header


## 7. `write_codebook` — `fileIO/WriteCodebook.m`

**What it does:** writes a codebook CSV in the format above.

(The original MATLAB function has a small bug where it references a `finalBarcodes` / `finalGenes` variable that isn't a parameter — it expects them in scope. The Python version takes them as proper arguments instead.)


In [ ]:
def write_codebook(codebook_path: str,
                   barcodes: np.ndarray,
                   bit_names: List[str],
                   names: List[str],
                   ids: List[str],
                   codebook_name: str = "M3E1",
                   version: str = "1.0",
                   verbose: bool = True) -> None:
    """Translate of fileIO/WriteCodebook.m (cleaned up).

    Parameters
    ----------
    codebook_path : str
        Output CSV path.
    barcodes : (M, N) ndarray of {0,1}
        M barcodes, each N bits.
    bit_names : list of str (length N)
        Names of the N readout bits.
    names : list of str (length M)
        Gene name per barcode.
    ids : list of str (length M)
        Transcript id per barcode.
    """
    barcodes = np.asarray(barcodes, dtype=np.uint8)
    M, N = barcodes.shape
    if len(names) != M or len(ids) != M:
        raise ValueError("len(names) and len(ids) must equal number of barcodes")
    if len(bit_names) != N:
        raise ValueError("len(bit_names) must equal barcode length")

    if not codebook_path.lower().endswith(".csv"):
        print(f"warning: codebook saved as non-csv: {codebook_path}")

    with open(codebook_path, "w") as f:
        f.write(f"version, {version}\n")
        f.write(f"codebook_name, {codebook_name}\n")
        f.write("bit_names, " + ", ".join(bit_names) + "\n")
        f.write("name, id, barcode\n")
        for i in range(M):
            bc_str = " ".join(str(int(b)) for b in barcodes[i])
            f.write(f"{names[i]}, {ids[i]}, {bc_str}\n")

    if verbose:
        print(f"-- Wrote codebook: {codebook_path}  ({M} barcodes, {N} bits)")


## 8. End-to-end demo — `example_scripts/code_construction_script.m`

This is the top-level demo from the original repository, ported line by line. It does two things:

1. **Build the MHD4 code** by generating Extended Hamming words on 11 data bits, then keeping only those with Hamming weight 4. Verify the minimum Hamming distance is 4.
2. **Build the MHD2 code** by listing every 14-bit binary word with exactly 4 on-bits. Verify the minimum Hamming distance is 2.

The output should match what the MATLAB version prints: the MHD4 code has 140 codewords of length 16 with minimum HD 4; the MHD2 code has C(14, 4) = 1001 codewords of length 14 with minimum HD 2.


In [ ]:
# ---- (1) MHD4 code ----------------------------------------------------
print("="*60)
print("Constructing MHD4 code (16 bits, Hamming weight 4)")
print("="*60)
num_data_bits = 11
EH_words, gen, num_parity = generate_extended_hamming_words(num_data_bits)
print(f"Extended Hamming code: {EH_words.shape[0]} words of length {EH_words.shape[1]}")
print(f"Generator matrix shape: {gen.shape}, parity bits: {num_parity}")

# Hamming weight = number of 1s in each codeword
hamming_weights = EH_words.sum(axis=1)
MHD4_words = EH_words[hamming_weights == 4]

print(f"\nConstructed {MHD4_words.shape[0]} MHD4 barcodes (length {MHD4_words.shape[1]})")
print(f"Hamming weights of MHD4 codewords: {sorted(set(MHD4_words.sum(axis=1).tolist()))}")

# Pairwise Hamming distance of MHD4
def pairwise_min_hd(words: np.ndarray) -> int:
    M = words.shape[0]
    min_d = words.shape[1]
    for i in range(M):
        for j in range(i+1, M):
            d = int(np.sum(words[i] != words[j]))
            if d < min_d:
                min_d = d
    return min_d

print(f"Minimum Hamming distance among MHD4 codewords: {pairwise_min_hd(MHD4_words)}")
print("(Expected: 4)")

# ---- (2) MHD2 code ----------------------------------------------------
print("\n" + "="*60)
print("Constructing MHD2 code (14 bits, exactly 4 on-bits)")
print("="*60)
num_bits = 14
on_bit_idx = list(combinations(range(num_bits), 4))
MHD2_words = np.zeros((len(on_bit_idx), num_bits), dtype=np.uint8)
for i, on_idx in enumerate(on_bit_idx):
    MHD2_words[i, list(on_idx)] = 1

print(f"Constructed {MHD2_words.shape[0]} MHD2 barcodes (length {MHD2_words.shape[1]})")
print(f"Hamming weights: {sorted(set(MHD2_words.sum(axis=1).tolist()))}")
print(f"Minimum Hamming distance among MHD2 codewords: {pairwise_min_hd(MHD2_words)}")
print("(Expected: 2)")


## 9. Self-tests

These cells verify the translated functions match the expected mathematical properties.


In [ ]:
# Test surrounding codewords
print("Test: generate_surrounding_codewords")
cw = np.array([1, 0, 1, 1, 0, 0, 1, 0, 0, 1, 0, 1, 1, 0, 0, 0], dtype=np.uint8)

n1 = generate_surrounding_codewords(cw, hamm_dist=1)
print(f"  HD=1 neighbours: {len(n1)}  (expected {cw.size} = number of single bit flips)")
assert len(n1) == cw.size
assert all(int(np.sum(neighbour != cw)) == 1 for neighbour in n1)

n2 = generate_surrounding_codewords(cw, hamm_dist=2)
expected_hd2 = cw.size * (cw.size - 1) // 2
print(f"  HD=2 neighbours: {len(n2)}  (expected C(16,2) = {expected_hd2})")
assert len(n2) == expected_hd2

# Test SECDED correctable
print("\nTest: secded_correctable_words")
sw = secded_correctable_words(cw)
print(f"  SECDED correctable words: {len(sw)}  (expected {cw.size})")
assert len(sw) == cw.size

# Test codebook_to_map
print("\nTest: codebook_to_map")
demo_codebook = [
    {"name": "GeneA", "id": "id_A", "barcode": "1011001000000000"},
    {"name": "GeneB", "id": "id_B", "barcode": "0100110000000000"},
]
m_exact = codebook_to_map(demo_codebook)
print(f"  exact map: {m_exact}")
assert len(m_exact) == 2

m_all = codebook_to_map(demo_codebook,
                        err_corr_func=secded_correctable_words,
                        map_contents="all")
print(f"  exact + correctable: {len(m_all)} entries (expected 2 + 2*16 = 34)")
assert len(m_all) == 34
print("\nAll self-tests passed ✓")


In [ ]:
# Round-trip test for load_codebook / write_codebook
print("Test: write_codebook → load_codebook round-trip")
import tempfile, os

tmp_path = os.path.join(tempfile.gettempdir(), "demo_codebook.csv")
n_genes, n_bits = 5, 16
demo_barcodes = MHD4_words[:n_genes]
demo_names = [f"Gene{i}" for i in range(n_genes)]
demo_ids = [f"id_{i:03d}" for i in range(n_genes)]
demo_bit_names = [f"RS{i:04d}" for i in range(n_bits)]

write_codebook(tmp_path, demo_barcodes, demo_bit_names,
               demo_names, demo_ids, codebook_name="DEMO", verbose=True)

loaded, hdr = load_codebook(tmp_path, verbose=True)
print(f"\nHeader: {hdr}")
print(f"First entry: {loaded[0]}")
assert len(loaded) == n_genes
assert loaded[0]["name"] == demo_names[0]
# Compare loaded barcode (string) to original (array)
loaded_arr = np.array([int(c) for c in loaded[0]["barcode"]], dtype=np.uint8)
assert np.array_equal(loaded_arr, demo_barcodes[0])
print("\nRound-trip OK ✓")
os.remove(tmp_path)


## 10. Summary

This notebook translates 8 of the 86 MATLAB files in ZhuangLab/MERFISH_analysis into self-contained Python:

| MATLAB file | Python function |
|---|---|
| `codes/GenerateExtendedHammingWords.m` | `generate_extended_hamming_words` |
| `codes/GenerateSurroundingCodewords.m` | `generate_surrounding_codewords` |
| `codes/SECDEDCorrectableWords.m` | `secded_correctable_words` |
| `codes/GenSECDED.m` | `gen_secded` |
| `codes/CodebookToMap.m` | `codebook_to_map` |
| `fileIO/LoadCodebook.m` | `load_codebook` |
| `fileIO/WriteCodebook.m` | `write_codebook` |
| `example_scripts/code_construction_script.m` | the demo cells (sections 8–9) |

The remaining 78 MATLAB files were not translated. They fall into three groups, each with a real reason:

- **`analysis/` (~20 files):** decode raw MERFISH fluorescence images. Requires raw `.dax` microscopy data we do not have. Not relevant to thesis work that starts from already-decoded gene expression.
- **`probe_construction/` (entire folder + `library_design_example.m`):** designs DNA oligonucleotide probes for new MERFISH experiments. Requires nucleotide sequence databases and BLAST. Designs experiments, not analyses them.
- **`fileIO/` proprietary readers, `startup/`, `deprecated/`:** read the Zhuang lab's custom binary formats (`.bin`, `.dax`, byte streams) or set up MATLAB paths. Not useful outside that lab's microscopy environment.

The translated subset is the part that operates on the same conceptual level as the thesis dataset: the 16-bit binary barcodes used to encode the 155 MERFISH genes Moffitt et al. used to label cells in the hypothalamic preoptic region.
